In [3]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI()

In [4]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [5]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [7]:
assistant.rag("How do I run Olama?")

'The provided context does not contain specific information on how to run Olama. If you have any references or commands related to running Olama, I can help interpret or clarify them based on the information available.'

In [8]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}


In [16]:
messages = [
    {"role": "user", "content": "How do I use Olama"}
]
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseOutputMessage(id='msg_08b3634fb032f8e4006a26e86d2af4819a95668e34a0bd2261', content=[ResponseOutputText(annotations=[], text='To use Olama, follow these steps:\n\n1. **Installation**: Download and install Olama from the official website or repository.\n\n2. **Set Up**: Follow the setup instructions, which may include configuring your environment or installing necessary dependencies.\n\n3. **Interface**: Open the application or interface, which could be a command line, GUI, or web platform.\n\n4. **Loading Models**: Load the language model you intend to use. This might involve specifying the model in the settings or command line.\n\n5. **Input**: Provide input text or prompts you want the model to respond to.\n\n6. **Execution**: Run the model to generate output based on your input.\n\n7. **Fine-Tuning**: Adjust settings or fine-tune the model parameters if necessary for better performance.\n\n8. **Output**: Collect and utilize the output as needed.\n\n9. **Documentation**: Refe

In [17]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

AttributeError: 'ResponseOutputMessage' object has no attribute 'arguments'

In [12]:
result_json

'[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2025."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you can only get a certificate if you finish the course with a \\"live\\" cohort.\\n\\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\\n

In [13]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [14]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course"}', call_id='call_rE5FHowikhw5dSJwqvdh6Rim', name='search', type='function_call', id='fc_0479970ad9f0f35f006a26e82a05b4819a8cc6e3a3cdc25d5e', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_rE5FHowikhw5dSJwqvdh6Rim',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2025."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zo

In [15]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes, you can still join the course! However, if you wish to receive a certificate, you need to submit your project while submissions are still being accepted. Feel free to start learning and submitting homework even without formal registration, as it’s not required to be on a specific list.'